# Pilotos v5b (ronda 2) — DistilBETO (IroSvA): nuevas técnicas contra sobreajuste y sobreconfianza

Notebook **independiente** de `DistilBETO_GroupNestedCV_5x3_TPE20_v4.ipynb`. Copia sin cambios sus celdas de librerías, datos, grupos, modelo y trainer, y agrega la sección de pilotos. **No ejecuta la Nested CV, no reoptimiza y no toca el test.**

Reentrena la configuración final v4 (`distilbeto_config_final.json`) en los mismos 3 Inner Folds y con las mismas semillas que el diagnóstico de convergencia, sin early stopping y con 4 evaluaciones por época. Cada piloto cambia **una sola cosa** respecto a P0:

| Piloto | Cambio | Respaldo |
|---|---|---|
| P0 | Ninguno (referencia, con la medición corregida) | — |
| P1 | Congelar embeddings + 2 de 6 capas (pasó la ronda 1) | Ingle et al. (2022); Merchant et al. (2020) |
| P4 | Sin pesos de clase (`loss_mode="standard"`) | Ablación de calibración |
| P5 | R-Drop, α = 1.0 | Liang et al. (NeurIPS 2021) |
| P6 | EMA de pesos, β = 0.99 (se evalúa con los pesos promediados) | Izmailov et al. (UAI 2018), promedio de pesos |
| P7 | Mixout p = 0.5 en el encoder (dropout del encoder a 0, como el código oficial) | Lee et al. (ICLR 2020) |
| P8 | LoRA r = 8, α = 16, dropout 0.05, LR 2e-4, sin LR decay (requiere `peft`) | Hu et al. (ICLR 2022) |
| P9 | Warmup 10 % **efectivo** (ver nota abajo) | Práctica estándar de fine-tuning (Devlin et al., 2019) |
| P10 | Propuesta del asesor: congelar embeddings y todo el encoder salvo el último tercio (BETO/RoBERTuito: últimas 4 de 12 capas; DistilBETO: últimas 2 de 6) + LR decay 0.9 | Howard & Ruder (2018); Sun et al. (2019) |
| P11 | Igual que P10 con LR decay 0.8 | Clark et al. (ICLR 2020, ELECTRA) |
| P12 | Dropout interno 0.2 (oculto y de atención; la v4 usa el 0.1 por defecto de los modelos) | Sugerencia del asesor; Srivastava et al. (JMLR 2014) |
| LC25/50/75 | P0 con 25 %, 50 % y 75 % del Inner Train (subconjuntos anidados por grupo) | Curva de aprendizaje |

Variantes: **MX y CU**. Total: 14 configuraciones × 2 variantes × 3 Inner Folds = **84 fine-tunings** (R-Drop cuesta ~2×).

**Nota importante sobre el warmup.** En transformers 5.x el argumento `warmup_ratio` ya no existe, y `argumentos_entrenamiento()` de la v4 descarta en silencio los argumentos que la versión instalada no acepta. Con transformers 5.17 (la versión de las tres corridas v4), **la v4 se entrenó sin warmup**, aunque el código dice `warmup_ratio=0.10`. P0 reproduce la v4 tal como se ejecutó (sin warmup) y P9 mide el efecto del warmup del 10 % aplicado de verdad (`warmup_steps=0.10`). En este notebook, si un argumento no es aceptado, la celda se detiene con un error en vez de descartarlo.

**Medición corregida tras la auditoría:**
- NLL **sin** pesos de clase ni label smoothing (la "pérdida" de la v4 estaba ponderada por clase).
- Brier, ECE con 10 bins de igual masa (y de igual ancho, como control), confianza media de los errores y % de errores con confianza > 0.9.
- Train evaluado sobre **todo** el Inner Train, en modo eval.
- F1 reportado de dos formas: máximo de la curva media y media de los máximos por fold.
- Logits de validación guardados, para temperature scaling con folds cruzados (T se ajusta en los otros 2 folds).
- Huella de configuración: un fold solo se reutiliza si su configuración coincide exactamente.

**Criterios fijados antes de ver resultados.** Una técnica es *candidata* para la v5 si:
1. No baja el F1 más de 0.01 frente a P0, con **ambas** métricas de F1; y
2. Reduce la brecha de F1 en al menos 0.03, **o** reduce la NLL de validación (en el mejor punto de F1) en al menos 0.02.

**Advertencia metodológica:** estos folds ya se usaron en la optimización final. Los pilotos son exploratorios; sus valores no son estimaciones de rendimiento.

**Cómo correrlo (Windows con GPU):**
1. (Opcional, para P8) `pip install peft` en el entorno. Si no está, P8 se omite automáticamente.
2. Copia este notebook a `notebooks/` (junto al v4), reinicia el kernel y ejecuta **Run All**.
3. Si se interrumpe, vuelve a ejecutar: reutiliza los folds terminados (si la huella coincide).
4. Revisa las líneas `[CHECK]` del primer fold de cada piloto: indican capas congeladas, % de parámetros entrenables, LR, R-Drop, EMA, Mixout y pesos de clase efectivos.

**Salidas** (en `data/resultados_distilbeto_grouped_5x3_tpe20_v4/`):
- `pilotos_v5b/<piloto>/`: curvas, logits y huella por fold.
- `distilbeto_pilotosb_resumen.csv`: tabla de decisión.
- `distilbeto_pilotosb_convergencia.csv`: todas las curvas.
- `distilbeto_pilotosb_tecnicas.png/.pdf` y `distilbeto_pilotosb_curva_aprendizaje.png/.pdf`: figuras.


## 1. Librerías y entorno (igual que v4)

In [ ]:
import os
import re

# Declarar antes de importar Transformers/Hugging Face para ocultar barras de descarga.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

import gc
import time
import json
import pickle
import html
import random
import tempfile
import shutil
import inspect
import warnings
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import optuna

from torch.utils.data import Dataset, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from IPython.display import display, HTML

import matplotlib

matplotlib.use("Agg")  # Evita abrir una ventana por gráfica durante cientos de fits.
import matplotlib.pyplot as plt

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.trainer_utils import get_last_checkpoint

try:
    from huggingface_hub.utils import disable_progress_bars

    disable_progress_bars()
except (ImportError, AttributeError):
    pass
transformers.logging.set_verbosity_error()
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)

VERSIONES = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "optuna": optuna.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 2. Configuración (igual que v4, salvo `RUN_DIR` y las banderas de ejecución)

In [ ]:
MODEL_NAME = "dccuchile/distilbert-base-spanish-uncased"

DATA_DIR = Path("../data/transformers")
RESULTS_DIR = Path("../data/resultados_distilbeto_grouped_5x3_tpe20_v4")
PROGRESS_DIR = RESULTS_DIR / "seguimiento"

TEXT_COLS = {"PYSENTIMIENTO": "MESSAGE_TRANSFORMER"}
LABEL_COL = "IS_IRONIC"
ID_COL = "ID"
ORIGINAL_COL = "MESSAGE"
GROUP_COL = "GROUP_ID"

MAX_LENGTH = 128
SEED = 42

# ------------------------------------------------------------
# "prueba" valida rápidamente todo el pipeline.
# "completo" = experimento definitivo 5×3 / TPE20.
# ------------------------------------------------------------
MODO = "completo"  # primero "prueba"; luego "completo" y reiniciar kernel

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS = 2
    N_STARTUP_TRIALS = 1
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS = 20
    N_STARTUP_TRIALS = 5
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

# RUN_TAG corto + subcarpetas compactas para Windows.
# v4: carpeta nueva (entrenamiento revisado tras el diagnóstico de convergencia).
RUN_TAG = "d5x3t20_v4"
RUN_DIR = PROGRESS_DIR / RUN_TAG / "pilotos_b"  # PILOTOS v5b: no toca la corrida v4
TMP_DIR = RUN_DIR / "tmp"
CHECKPOINT_DIR = RUN_DIR / "ckpt"
FIT_STATE_DIR = RUN_DIR / "state"
CURVE_DIR = RUN_DIR / "curves"
GRAPH_DIR = RUN_DIR / "graphs"
AUDIT_DIR = RUN_DIR / "audit"

for carpeta in [
    RESULTS_DIR,
    PROGRESS_DIR,
    RUN_DIR,
    TMP_DIR,
    CHECKPOINT_DIR,
    FIT_STATE_DIR,
    CURVE_DIR,
    GRAPH_DIR,
    AUDIT_DIR,
]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Flujo recomendado (sección 18): opcionalmente una corrida corta con MODO="prueba";
# luego MODO="completo". Nested CV y optimización final usan solo train y corren seguidas;
# el test oficial requiere además CONFIRMACION_TEST.
EJECUTAR_NESTED = False  # pilotos: no se ejecuta la Nested CV
EJECUTAR_OPTIMIZACION_FINAL = False  # pilotos: no se reoptimiza
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

SEMILLAS_FINALES = [42, 52, 62, 72, 82]
CLEAN_COMPLETED_CHECKPOINTS = True

# Seguimiento: panel de progreso común a todos los notebooks de modelos.
NOMBRE_EXPERIMENTO = "DistilBETO"
HEARTBEAT_SEGUNDOS = 30
# Pitido de Windows al terminar o detenerse una etapa larga (False para desactivarlo).
AVISO_SONORO = True

# Optuna TPE: 5 trials iniciales aleatorios y 15 adaptativos en modo completo.
# Espacio común a BETO, DistilBETO y RoBERTuito.
SEARCH_SPACE = {
    "preprocessing": ["PYSENTIMIENTO"],
    "learning_rate": [1e-5, 2e-5],  # v3: 2e-5..5e-5 (memorizaba en ~100 pasos)
    "batch_size": [16],  # fijo: probado en los pilotos B y C
    "num_train_epochs": [4, 6],  # máximo; decide el early stopping
    "weight_decay": [0.01, 0.1],
    "loss_mode": ["standard", "balanced"],
    "classifier_dropout": [0.1, 0.2, 0.3],
    "label_smoothing": [0.1],  # fijo (piloto B)
    "lr_decay": [0.9],  # fijo (piloto B)
}

# Early stopping de los Inner Folds: épocas sin mejora de F1 antes de detener.
PACIENCIA_EARLY_STOPPING = 2

SEARCH_SPACE_REQUERIDO = {
    "preprocessing",
    "learning_rate",
    "batch_size",
    "num_train_epochs",
    "weight_decay",
    "loss_mode",
    "classifier_dropout",
    "label_smoothing",
    "lr_decay",
}
faltantes_search_space = SEARCH_SPACE_REQUERIDO - set(SEARCH_SPACE)
if faltantes_search_space:
    raise KeyError(f"SEARCH_SPACE incompleto. Faltan: {sorted(faltantes_search_space)}")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

fits_teoricos = len(VARIANTES) * N_OUTER * (N_TRIALS * N_INNER + 1)

print(f"[CONFIG][DistilBETO] modo={MODO} | variantes={VARIANTES}")
print(
    f"[CONFIG][DistilBETO] Nested={N_OUTER}×{N_INNER} | "
    f"trials={N_TRIALS} | startup={N_STARTUP_TRIALS}"
)
print(
    f"[CONFIG][DistilBETO] fine-tunings teóricos={fits_teoricos} | "
    f"nested_activo={EJECUTAR_NESTED}"
)
print(f"[CONFIG][DistilBETO] RUN_DIR={RUN_DIR.resolve()}")
print("[CONFIG][DistilBETO] SEARCH_SPACE OK:", sorted(SEARCH_SPACE))

# Comprobación preventiva de longitud de ruta en Windows (peor caso del espacio).
_ruta_ckpt_ejemplo = (
    CHECKPOINT_DIR / "mx" / "outer1" / "P_l2_b16_e6_w10_b_d3_s1_r9" / "inner3" / "checkpoint-99999"
).resolve()

print(
    f"[CONFIG][DistilBETO] longitud ruta checkpoint ejemplo="
    f"{len(str(_ruta_ckpt_ejemplo))} caracteres"
)

if os.name == "nt" and len(str(_ruta_ckpt_ejemplo)) >= 230:
    raise RuntimeError(
        "La ruta prevista para checkpoints sigue siendo demasiado larga en Windows. "
        "Mueve el repositorio a una ruta más corta antes de entrenar."
    )

## 3. Carga del train y verificación del preprocesamiento (igual que v4)

In [ ]:
MANIFEST_PATH = DATA_DIR / "manifest_preprocesamiento_transformers.csv"
PIPELINE_VERSION_ESPERADA = "transformers_comun_v2"


def cargar_manifest_preprocesamiento():
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"Falta {MANIFEST_PATH}. Ejecuta primero "
            "`01_preprocesamiento_dl_transformers.ipynb`."
        )

    manifest = pd.read_csv(MANIFEST_PATH)

    requeridas = {"pipeline_version", "variante", "train_final"}
    faltantes = requeridas - set(manifest.columns)
    if faltantes:
        raise ValueError(f"Manifest incompleto; faltan columnas: {sorted(faltantes)}")

    versiones = set(manifest["pipeline_version"].astype(str))
    if versiones != {PIPELINE_VERSION_ESPERADA}:
        raise ValueError(
            f"Versión de preprocessing inesperada: {versiones}. "
            f"Se esperaba {PIPELINE_VERSION_ESPERADA}."
        )

    return manifest


MANIFEST_PREPROCESAMIENTO = cargar_manifest_preprocesamiento()


def validar_archivo_train_con_manifest(variante):
    ruta = DATA_DIR / f"train_transformers_{variante}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecuta primero el preprocessing Transformer.")

    fila = MANIFEST_PREPROCESAMIENTO[MANIFEST_PREPROCESAMIENTO["variante"].astype(str) == variante]
    if len(fila) != 1:
        raise ValueError(f"{variante}: se esperaba exactamente una fila en el manifest.")

    return ruta, fila.iloc[0]


def cargar_train(variante):
    ruta, manifest_row = validar_archivo_train_con_manifest(variante)
    df = pd.read_csv(ruta)

    requeridas = [ID_COL, ORIGINAL_COL, LABEL_COL, *TEXT_COLS.values()]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{variante}: faltan columnas {faltantes}")

    df[ID_COL] = df[ID_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    for col in [ORIGINAL_COL, *TEXT_COLS.values()]:
        df[col] = df[col].fillna("").astype(str)

    if df[ID_COL].duplicated().any():
        raise ValueError(f"{variante}: IDs duplicados en el train FINAL.")

    if not set(df[LABEL_COL].unique()).issubset({0, 1}):
        raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

    vacios = {
        col: int(df[col].eq("").sum())
        for col in [ORIGINAL_COL, *TEXT_COLS.values()]
        if df[col].eq("").any()
    }
    if vacios:
        raise ValueError(
            f"{variante}: entradas vacías en el train final {vacios}. "
            "Vuelve a ejecutar `01_preprocesamiento_dl_transformers.ipynb` "
            "(MESSAGE_TRANSFORMER requiere `pysentimiento` instalado)."
        )

    esperado = int(manifest_row["train_final"])
    if len(df) != esperado:
        raise ValueError(f"{variante}: train={len(df)} pero manifest={esperado}.")

    return df.reset_index(drop=True)


for variante in ["mx", "es", "cu"]:
    tmp = cargar_train(variante)
    print(
        f"[DATA][DistilBETO][{variante.upper()}] "
        f"n={len(tmp)} | "
        f"clases={tmp[LABEL_COL].value_counts().sort_index().to_dict()} | "
        "manifest=OK"
    )

## 4. Tokenizador y datos (igual que v4)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Tokenizer:", tokenizer.__class__.__name__)
print("do_lower_case:", getattr(tokenizer, "do_lower_case", "gestionado por tokenizer"))
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None
)


class DatasetTokenizado(Dataset):
    def __init__(self, textos, etiquetas):
        self.encoded = tokenizer(
            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False
        )
        self.labels = np.asarray(etiquetas, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encoded.items()}
        item["labels"] = int(self.labels[i])
        return item


def cache_dataset(df):
    y = df[LABEL_COL].to_numpy(dtype=np.int64)
    return {prep: DatasetTokenizado(df[col].tolist(), y) for prep, col in TEXT_COLS.items()}, y

### 4.1 `GROUP_ID` (igual que v4: mismos grupos, mismos folds)

In [ ]:
from collections import defaultdict


def colapsar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def clave_tokens(ids):
    """Representación directa de los input_ids, sin hashes."""
    return ",".join(map(str, ids))


def construir_grupos_distilbeto(df, variante, tokenizador=tokenizer, max_length=MAX_LENGTH):
    """Agrupa entradas equivalentes para evitar que crucen train/validación.

    Agrupación propia del modelo: texto original, cada representación de
    entrada (TEXT_COLS) y su secuencia tokenizada (truncada a MAX_LENGTH).
    """
    trabajo = df.copy().reset_index(drop=True)
    trabajo["KEY_ORIGINAL"] = trabajo[ORIGINAL_COL].map(colapsar_espacios)
    for prep, columna in TEXT_COLS.items():
        trabajo[f"KEY_{prep}"] = trabajo[columna].map(colapsar_espacios)

    for prep, columna in TEXT_COLS.items():
        ids = tokenizador(
            trabajo[columna].astype(str).tolist(),
            truncation=True,
            max_length=max_length,
            padding=False,
        )["input_ids"]
        trabajo[f"KEY_TOK_{prep}"] = [clave_tokens(x) for x in ids]

    representaciones = ["ORIGINAL", *TEXT_COLS, *[f"TOK_{p}" for p in TEXT_COLS]]
    key_cols = [f"KEY_{r}" for r in representaciones]
    uf = UnionFind(len(trabajo))

    for col in key_cols:
        first_seen = {}
        for idx, key in enumerate(trabajo[col].tolist()):
            if not key:
                continue
            if key in first_seen:
                uf.union(idx, first_seen[key])
            else:
                first_seen[key] = idx

    componentes = defaultdict(list)
    for idx in range(len(trabajo)):
        componentes[uf.find(idx)].append(idx)

    group_ids = [None] * len(trabajo)

    componentes_ordenados = sorted(componentes.values(), key=lambda miembros: min(miembros))

    for numero_grupo, miembros in enumerate(componentes_ordenados, start=1):
        gid = f"{variante}_G{numero_grupo:04d}"
        for idx in miembros:
            group_ids[idx] = gid
    trabajo[GROUP_COL] = group_ids

    resumen = []
    for nombre, col in zip(representaciones, key_cols):
        sizes = trabajo.groupby(col).size()
        repetidos = sizes[sizes > 1]
        resumen.append(
            {
                "variante": variante,
                "representacion": nombre,
                "filas_en_repeticiones": int(trabajo[col].isin(repetidos.index).sum()),
                "grupos_repetidos": int(len(repetidos)),
            }
        )

    gstats = trabajo.groupby(GROUP_COL)[LABEL_COL].agg(["size", "nunique"])
    conflictivos = set(gstats[(gstats["size"] > 1) & (gstats["nunique"] > 1)].index)
    detalle_conflictos = trabajo[trabajo[GROUP_COL].isin(conflictivos)].copy()

    resumen = pd.DataFrame(resumen)
    resumen["n_train"] = len(trabajo)
    resumen["n_group_id"] = trabajo[GROUP_COL].nunique()
    resumen["max_group_size"] = int(trabajo.groupby(GROUP_COL).size().max())
    resumen["grupos_con_labels_mixtos"] = len(conflictivos)
    return trabajo, resumen, detalle_conflictos


TRAIN_AGRUPADO = {}
AUDITORIA_GRUPOS = []
DETALLES_CONFLICTOS = []

# Se construyen los grupos de las tres variantes antes de una corrida larga.
for variante in ["mx", "es", "cu"]:
    df_audit = cargar_train(variante)
    agrupado, resumen, conflictos = construir_grupos_distilbeto(df_audit, variante)
    TRAIN_AGRUPADO[variante] = agrupado
    AUDITORIA_GRUPOS.append(resumen)
    if not conflictos.empty:
        conflictos["variante"] = variante
        DETALLES_CONFLICTOS.append(conflictos)

df_auditoria_grupos = pd.concat(AUDITORIA_GRUPOS, ignore_index=True)
df_auditoria_grupos.to_csv(RUN_DIR / "auditoria_grupos_train.csv", index=False)
display(df_auditoria_grupos)

if DETALLES_CONFLICTOS:
    df_conflictos_grupo = pd.concat(DETALLES_CONFLICTOS, ignore_index=True)
    df_conflictos_grupo.to_csv(RUN_DIR / "auditoria_grupos_labels_mixtos.csv", index=False)

for variante in ["mx", "es", "cu"]:
    df_g = TRAIN_AGRUPADO[variante]
    n_mixtos = int(df_g.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())
    print(
        f"[AUDIT][DistilBETO][{variante.upper()}] n={len(df_g)} | "
        f"groups={df_g[GROUP_COL].nunique()} | "
        f"max_group={df_g.groupby(GROUP_COL).size().max()} | labels_mixtos={n_mixtos}"
    )

    if n_mixtos > 0:
        print(
            f"  ADVERTENCIA: {n_mixtos} GROUP_ID contienen etiquetas mixtas. "
            "No es data leakage porque el grupo permanece íntegro dentro de un fold, "
            "pero representa ambigüedad/ruido de etiqueta y debe documentarse."
        )

### 4.2 Pre-flight (igual que v4; exige GPU)

In [ ]:
def comprobar_preflight_distilbeto():
    print("=" * 78)
    print("PRE-FLIGHT — DISTILBETO GROUPED NESTED CV")
    print("=" * 78)

    if MODO == "completo":
        esperado = (N_OUTER, N_INNER, N_TRIALS, N_STARTUP_TRIALS)

        if esperado != (5, 3, 20, 5):
            raise ValueError(
                f"Configuración inesperada: {esperado}; " "se esperaba 5×3, 20 trials, startup=5."
            )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA no disponible. Verifica entorno/PyTorch antes de entrenar DistilBETO."
        )

    for variante in ["mx", "es", "cu"]:
        _, manifest_row = validar_archivo_train_con_manifest(variante)

        df = TRAIN_AGRUPADO[variante]

        if len(df) != int(manifest_row["train_final"]):
            raise ValueError(f"{variante}: train agrupado no coincide " "con el manifest.")

        if df[ID_COL].duplicated().any():
            raise ValueError(f"{variante}: IDs duplicados.")

        if df[GROUP_COL].isna().any():
            raise ValueError(f"{variante}: GROUP_ID nulo.")

        if not set(df[LABEL_COL].unique()).issubset({0, 1}):
            raise ValueError(f"{variante}: etiquetas fuera de 0/1.")

        n_mixtos = int(df.groupby(GROUP_COL)[LABEL_COL].nunique().gt(1).sum())

        if n_mixtos:
            print(
                f"[AUDIT][DistilBETO][{variante.upper()}] "
                f"ADVERTENCIA: {n_mixtos} GROUP_ID "
                "con etiquetas mixtas; se documentan "
                "como ruido/ambigüedad."
            )

    print("Preprocessing:", PIPELINE_VERSION_ESPERADA)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Nested: {N_OUTER}×{N_INNER} | Optuna={N_TRIALS} | startup={N_STARTUP_TRIALS}")
    print("Train final:", {v: len(TRAIN_AGRUPADO[v]) for v in ["mx", "es", "cu"]})
    print("PRE-FLIGHT DE DATOS: OK")


comprobar_preflight_distilbeto()

## 5. Métricas, modelo y trainer (igual que v4)

In [ ]:
def establecer_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def metricas(prediction):
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    y_pred = np.asarray(logits).argmax(axis=-1)
    return {
        "f1_macro": f1_score(labels, y_pred, average="macro"),
        "accuracy": accuracy_score(labels, y_pred),
        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),
    }


def pesos_balanceados(y):
    classes = np.array([0, 1])
    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))
    return torch.tensor(w, dtype=torch.float32)


def crear_modelo(seed, classifier_dropout=0.2):
    establecer_semilla(seed)
    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(
        log
    ), redirect_stderr(log):
        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
        if config.model_type != "distilbert":
            raise ValueError(f"Se esperaba arquitectura DistilBERT, recibida: {config.model_type}")
        # DistilBERT guarda el dropout de la cabeza de clasificación en `seq_classif_dropout`.
        config.seq_classif_dropout = float(classifier_dropout)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    return model


class ClasificadorTrainer(Trainer):
    """v4: pérdida ponderada opcional + label smoothing (solo en train) + LR decay por capas."""

    def __init__(self, *args, class_weights=None, label_smoothing=0.0, lr_decay=1.0, **kwargs):
        self.class_weights = class_weights
        self.label_smoothing = float(label_smoothing)
        self.lr_decay = float(lr_decay)
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        weights = (
            self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        )
        # Smoothing SOLO al entrenar: la pérdida de validación sigue siendo entropía cruzada estándar.
        ls = self.label_smoothing if model.training else 0.0
        loss = F.cross_entropy(outputs.logits, labels, weight=weights, label_smoothing=ls)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self, *args, **kwargs):
        if self.optimizer is not None:
            return self.optimizer
        if self.lr_decay >= 1.0:
            return super().create_optimizer(*args, **kwargs)
        # Capas: BERT/RoBERTa `encoder.layer.N`, DistilBERT `transformer.layer.N`.
        patron = re.compile(r"\.layer\.(\d+)\.")
        n_capas = 1 + max(
            int(m.group(1)) for n, _ in self.model.named_parameters() if (m := patron.search(n))
        )
        base_lr, wd = self.args.learning_rate, self.args.weight_decay
        grupos = {}
        for nombre, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            m = patron.search(nombre)
            if m:
                profundidad = n_capas - int(m.group(1))  # última capa = 1
            elif "embeddings" in nombre:
                profundidad = n_capas + 1  # embeddings = LR más bajo
            else:
                profundidad = 0  # cabeza de clasificación / pooler = LR base
            sin_wd = nombre.endswith("bias") or "LayerNorm" in nombre or "layer_norm" in nombre
            clave = (base_lr * self.lr_decay**profundidad, 0.0 if sin_wd else wd)
            grupos.setdefault(clave, []).append(p)
        self.optimizer = torch.optim.AdamW(
            [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in grupos.items()]
        )
        return self.optimizer


class ParadaTemprana(TrainerCallback):
    """Early stopping por F1-Macro de validación sin guardar checkpoints (Inner Folds)."""

    def __init__(self, paciencia=2):
        self.paciencia, self.mejor, self.sin_mejora = int(paciencia), -np.inf, 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        f1 = (metrics or {}).get("eval_f1_macro")
        if f1 is None:
            return control
        if f1 > self.mejor:
            self.mejor, self.sin_mejora = f1, 0
        else:
            self.sin_mejora += 1
            if self.sin_mejora >= self.paciencia:
                control.should_training_stop = True
        return control


class DetenerEnEpoca(TrainerCallback):
    """Refit/test con el MISMO scheduler que la búsqueda: se detiene en la época elegida."""

    def __init__(self, epocas):
        self.epocas = int(epocas)

    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch is not None and state.epoch >= self.epocas - 1e-6:
            control.should_training_stop = True
        return control


## 6. Argumentos de entrenamiento (igual que v4)

In [ ]:
def argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):
    params = {
        "output_dir": str(carpeta),
        "learning_rate": cfg["learning_rate"],
        "per_device_train_batch_size": cfg["batch_size"],
        "per_device_eval_batch_size": cfg["batch_size"],
        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],
        "weight_decay": cfg["weight_decay"],
        "warmup_ratio": 0.10,
        "optim": "adamw_torch",
        # Trainer recoge loss por época en state.log_history;
        # PrinterCallback se elimina para no saturar VS Code.
        "logging_strategy": "epoch",
        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.
        "load_best_model_at_end": bool(con_validacion),
        "save_total_limit": 2,  # último + mejor (si son diferentes)
        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.
        "save_safetensors": True,
        "report_to": "none",
        "disable_tqdm": True,
        # Necesario para que EarlyStoppingCallback conserve su contador
        # al reanudar desde un checkpoint.
        "restore_callback_states_from_checkpoint": True,
        "dataloader_num_workers": 0,
        "seed": seed,
        "data_seed": seed,
        "bf16": USE_BF16,
        "fp16": USE_FP16,
    }
    if con_validacion:
        # v4 — Inner Folds sin checkpoints: el modelo interno se descarta y F1/mejor época
        # se leen de la curva. El early stopping lo hace ParadaTemprana.
        params.update({"save_strategy": "no", "load_best_model_at_end": False})
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        params["eval_strategy"] = "epoch" if con_validacion else "no"
    elif "evaluation_strategy" in sig:
        params["evaluation_strategy"] = "epoch" if con_validacion else "no"
    else:
        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")
    return TrainingArguments(**{k: v for k, v in params.items() if k in sig})


def mejor_epoca(trainer, max_epochs):
    evaluaciones = [x for x in trainer.state.log_history if "eval_f1_macro" in x and "epoch" in x]
    if not evaluaciones:
        raise RuntimeError("No se encontró F1 de validación por época.")
    mejor = max(evaluaciones, key=lambda x: x["eval_f1_macro"])
    return int(np.clip(round(float(mejor["epoch"])), 1, max_epochs))


def predecir(trainer, dataset):
    logits = trainer.predict(dataset).predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    return np.argmax(logits, axis=-1)

## 7. Utilidades (copiadas de la sección 7 de la v4)

In [ ]:
def entrenador_silencioso(trainer):
    trainer.remove_callback(PrinterCallback)
    trainer.remove_callback(ProgressCallback)
    return trainer


def csv_atomico(df, ruta):
    ruta = Path(ruta)
    tmp = ruta.with_name(ruta.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)


## 8. Pilotos v5b

Revisa en las líneas `[CHECK]`: P1 debe decir `capas congeladas=[0, 1]`; P4 `loss=standard`; P5 `R-Drop=1.0`; P6 `EMA=0.99`; P7 `MixLinear=...` (> 0); P8 `entrenables=` alrededor de 1 % y `LR=0.0002`; P10/P11 `capas congeladas=[0..3]` y `LRD=0.9`/`0.8`; P12 `dropouts internos a 0.2=` > 0; P9 `warmup=` > 0 pasos (P0 debe decir `warmup=0 pasos`); LC25 `n_train` ≈ 25 % del de P0.


In [ ]:
# ============================================================
# PILOTOS v5b — ronda 2: nuevas técnicas + medición corregida
# Solo train oficial: mismos 3 Inner Folds y mismas semillas que el diagnóstico v4.
# Sin early stopping, 4 evaluaciones por época. NO toca el test.
# Exploratorio: estos folds ya se usaron en la optimización final (sesgo de selección),
# así que los resultados NO son estimaciones de rendimiento; solo deciden qué probar.
#
# Medición corregida (auditoría):
#   - NLL SIN pesos de clase y sin label smoothing (la "pérdida" de v4 era ponderada)
#   - Brier, ECE con 10 bins de igual masa (y de igual ancho), confianza de los errores
#   - Train evaluado sobre TODO el Inner Train, en modo eval
#   - F1 por "máximo de la curva media" y por "media de los máximos por fold"
#   - Logits de validación guardados -> temperature scaling con folds cruzados
#   - Huella de configuración: solo se reutiliza un CSV si la configuración coincide
# ============================================================
import math
import time
import hashlib

VARIANTES_PILOTO = ["mx", "cu"]
PILOTOS_A_CORRER = ["P0", "P1", "P4", "P5", "P6", "P7", "P8", "P9", "P10", "P11", "P12", "LC25", "LC50", "LC75"]
EVALS_POR_EPOCA = 4
REHACER = False
# Criterios fijados ANTES de ver resultados
UMBRAL_F1 = 0.01        # no inferioridad: ΔF1 >= -0.01 con AMBAS métricas de F1
UMBRAL_BRECHA = 0.03    # mejora si la brecha de F1 baja al menos 0.03 ...
UMBRAL_NLL = 0.02       # ... o la NLL de validación (en el mejor F1) baja al menos 0.02

_cfg_hf = AutoConfig.from_pretrained(MODEL_NAME)
TIPO_MODELO = _cfg_hf.model_type
N_CAPAS = int(getattr(_cfg_hf, "num_hidden_layers", 0) or getattr(_cfg_hf, "n_layers"))
CAPAS_25 = max(1, round(N_CAPAS * 0.25))
# Propuesta del asesor: entrenar solo el último tercio del encoder (4 de 12 capas;
# en DistilBETO, 2 de 6) + cabeza, con LR decay por capas.
CAPAS_ENTRENABLES_ASESOR = max(1, round(N_CAPAS / 3))
CAPAS_CONGELADAS_ASESOR = N_CAPAS - CAPAS_ENTRENABLES_ASESOR
LORA_MODULOS = ["q_lin", "v_lin"] if TIPO_MODELO == "distilbert" else ["query", "value"]
LORA_GUARDAR = ["classifier", "pre_classifier"] if TIPO_MODELO == "distilbert" else ["classifier"]

# Cada piloto cambia UNA cosa respecto a P0 (configuración final v4).
PILOTOS = {
    "P0": {"nombre": "v4 (referencia)"},
    "P1": {"nombre": f"congelar emb + {CAPAS_25}/{N_CAPAS} capas", "congelar_capas": CAPAS_25},
    "P4": {"nombre": "sin pesos de clase", "loss_mode": "standard"},
    "P5": {"nombre": "R-Drop α=1.0", "rdrop_alpha": 1.0},
    "P6": {"nombre": "EMA de pesos β=0.99", "ema_beta": 0.99},
    "P7": {"nombre": "Mixout p=0.5", "mixout_p": 0.5},
    "P8": {"nombre": "LoRA r=8 α=16 (LR 2e-4)", "lora_r": 8, "lora_alpha": 16,
           "lora_dropout": 0.05, "learning_rate": 2e-4, "lr_decay": 1.0},
    "P9": {"nombre": "warmup 10 % efectivo", "warmup": 0.10},
    "P10": {"nombre": f"solo últimas {CAPAS_ENTRENABLES_ASESOR}/{N_CAPAS} capas + LRD 0.9 (asesor)",
            "congelar_capas": CAPAS_CONGELADAS_ASESOR, "lr_decay": 0.9},
    "P11": {"nombre": f"solo últimas {CAPAS_ENTRENABLES_ASESOR}/{N_CAPAS} capas + LRD 0.8 (asesor)",
            "congelar_capas": CAPAS_CONGELADAS_ASESOR, "lr_decay": 0.8},
    "P12": {"nombre": "dropout interno 0.2 (oculto + atención; asesor)", "dropout_interno": 0.2},
    "LC25": {"nombre": "P0 con 25 % del Inner Train", "fraccion_train": 0.25},
    "LC50": {"nombre": "P0 con 50 % del Inner Train", "fraccion_train": 0.50},
    "LC75": {"nombre": "P0 con 75 % del Inner Train", "fraccion_train": 0.75},
}
ES_CURVA_APRENDIZAJE = {"LC25", "LC50", "LC75"}

# IMPORTANTE (hallazgo de auditoría): en transformers >= 5, `warmup_ratio` ya no existe y
# `argumentos_entrenamiento` de la v4 lo descartaba en silencio -> la v4 se entrenó SIN warmup.
# P0 reproduce la v4 tal como se ejecutó (warmup 0); P9 mide el efecto del warmup 10 % real.
WARMUP_V4_EFECTIVO = 0.0 if "warmup_ratio" not in inspect.signature(TrainingArguments.__init__).parameters else 0.10
print(f"[PILOTOS-B] warmup efectivo de la v4 en este entorno = {WARMUP_V4_EFECTIVO}")

BASE = NOMBRE_EXPERIMENTO.lower()
PILOTOS_DIR = RESULTS_DIR / "pilotos_v5b"
PATRON_CAPA = re.compile(r"\.layer\.(\d+)\.")

try:
    import peft
    VERSION_PEFT = peft.__version__
except ImportError:
    VERSION_PEFT = None
    if "P8" in PILOTOS_A_CORRER:
        print("⚠️  `peft` no está instalado: P8 (LoRA) se omitirá. Para incluirlo: pip install peft")
        PILOTOS_A_CORRER = [p for p in PILOTOS_A_CORRER if p != "P8"]

print(f"[PILOTOS-B][{NOMBRE_EXPERIMENTO}] tipo={TIPO_MODELO} | capas={N_CAPAS} | "
      f"variantes={VARIANTES_PILOTO} | pilotos={PILOTOS_A_CORRER} | peft={VERSION_PEFT}")


# ------------------------------------------------------------------
# Técnicas
# ------------------------------------------------------------------
def congelar_capas(model, n_capas):
    """Congela embeddings + capas Transformer 0..n_capas-1 (Merchant et al., 2020)."""
    capas = set()
    for nombre, p in model.named_parameters():
        m = PATRON_CAPA.search(nombre)
        if "embeddings" in nombre or (m and int(m.group(1)) < n_capas):
            p.requires_grad = False
            if m:
                capas.add(int(m.group(1)))
    return sorted(capas)


class MixLinear(torch.nn.Module):
    """Mixout (Lee et al., ICLR 2020): con prob. p cada peso se reemplaza por su valor
    preentrenado y se reescala para conservar la esperanza (código oficial de los autores)."""

    def __init__(self, lineal, p):
        super().__init__()
        self.p = float(p)
        self.in_features, self.out_features = lineal.in_features, lineal.out_features
        self.weight = lineal.weight
        self.bias = lineal.bias
        self.register_buffer("target", lineal.weight.detach().clone())

    def forward(self, x):
        w = self.weight
        if self.training and self.p > 0:
            keep = torch.empty_like(w).bernoulli_(1.0 - self.p)
            w = (keep * w + (1.0 - keep) * self.target - self.p * self.target) / (1.0 - self.p)
        return F.linear(x, w, self.bias)


def aplicar_mixout(model, p):
    """Reemplaza los nn.Linear del encoder por MixLinear y, como el código oficial, pone a 0
    el dropout del encoder y los embeddings. La cabeza de clasificación no se toca."""
    reemplazos, dropouts = 0, 0
    for nombre_padre, padre in list(model.named_modules()):
        for nombre, hijo in list(padre.named_children()):
            completo = f"{nombre_padre}.{nombre}." if nombre_padre else f"{nombre}."
            en_encoder = bool(PATRON_CAPA.search(completo)) or "embeddings" in completo
            if not en_encoder:
                continue
            if isinstance(hijo, torch.nn.Linear):
                setattr(padre, nombre, MixLinear(hijo, p))
                reemplazos += 1
            elif isinstance(hijo, torch.nn.Dropout):
                hijo.p = 0.0
                dropouts += 1
    return reemplazos, dropouts


def fijar_dropout_interno(model, p):
    """Fija el dropout de embeddings y encoder (hidden_dropout_prob y attention_probs_dropout_prob;
    en DistilBERT, dropout y attention_dropout). Equivale a cambiarlos en el config antes de
    cargar el modelo: en transformers 5 la atención lee `self.dropout.p` en cada forward."""
    n = 0
    for nombre, modulo in model.named_modules():
        completo = f"{nombre}."
        if isinstance(modulo, torch.nn.Dropout) and (PATRON_CAPA.search(completo) or "embeddings" in completo):
            modulo.p = float(p)
            n += 1
    return n


def aplicar_lora(model, d):
    from peft import LoraConfig, TaskType, get_peft_model
    cfg_lora = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=int(d["lora_r"]), lora_alpha=int(d["lora_alpha"]),
        lora_dropout=float(d["lora_dropout"]), target_modules=LORA_MODULOS,
        modules_to_save=LORA_GUARDAR,
    )
    return get_peft_model(model, cfg_lora)


class CallbackEMA(TrainerCallback):
    """EMA de los pesos entrenables. Durante el warmup copia los pesos; después promedia.
    Las evaluaciones usan los pesos EMA (ver TrainerPilotos.evaluate)."""

    def __init__(self, beta):
        self.beta, self.sombra, self.warmup = float(beta), None, 0

    def on_train_begin(self, args, state, control, model=None, **kwargs):
        self.warmup = int(args.get_warmup_steps(state.max_steps))

    @torch.no_grad()
    def on_step_end(self, args, state, control, model=None, **kwargs):
        params = {n: p for n, p in model.named_parameters() if p.requires_grad}
        if self.sombra is None or state.global_step <= self.warmup:
            self.sombra = {n: p.detach().float().clone() for n, p in params.items()}
        else:
            for n, p in params.items():
                self.sombra[n].mul_(self.beta).add_(p.detach().float(), alpha=1.0 - self.beta)
        return control


class TrainerPilotos(ClasificadorTrainer):
    """ClasificadorTrainer v4 + R-Drop opcional + evaluación con pesos EMA."""

    def __init__(self, *args, rdrop_alpha=0.0, callback_ema=None, **kwargs):
        self.rdrop_alpha = float(rdrop_alpha)
        self.callback_ema = callback_ema
        self.prefijo_eval = "eval"
        super().__init__(*args, **kwargs)
        if callback_ema is not None:
            self.add_callback(callback_ema)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        if not (model.training and self.rdrop_alpha > 0):
            return super().compute_loss(model, inputs, return_outputs, num_items_in_batch, **kwargs)
        # R-Drop (Liang et al., NeurIPS 2021): dos pasadas con dropout distinto.
        # L = media(CE1, CE2) + α · ½[KL(p1‖p2) + KL(p2‖p1)], KL promediada por ejemplo.
        labels = inputs["labels"]
        x = {k: torch.cat([v, v], dim=0) for k, v in inputs.items() if k != "labels"}
        l1, l2 = model(**x).logits.chunk(2, dim=0)
        w = self.class_weights.to(l1.device) if self.class_weights is not None else None
        ce = 0.5 * (
            F.cross_entropy(l1, labels, weight=w, label_smoothing=self.label_smoothing)
            + F.cross_entropy(l2, labels, weight=w, label_smoothing=self.label_smoothing)
        )
        lp1, lp2 = F.log_softmax(l1.float(), dim=-1), F.log_softmax(l2.float(), dim=-1)
        kl = 0.5 * (
            F.kl_div(lp1, lp2, log_target=True, reduction="batchmean")
            + F.kl_div(lp2, lp1, log_target=True, reduction="batchmean")
        )
        loss = ce + self.rdrop_alpha * kl
        return (loss, {"logits": l1}) if return_outputs else loss

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        conjunto = self.eval_dataset if eval_dataset is None else eval_dataset
        if isinstance(conjunto, dict):  # Trainer vuelve a llamar evaluate() por cada clave
            return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)
        self.prefijo_eval = metric_key_prefix
        sombra = self.callback_ema.sombra if self.callback_ema is not None else None
        if not sombra:
            return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)
        params = dict(self.model.named_parameters())
        respaldo = {n: params[n].detach().clone() for n in sombra}
        with torch.no_grad():
            for n, v in sombra.items():
                params[n].copy_(v.to(params[n].dtype))
        try:
            return super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)
        finally:
            with torch.no_grad():
                for n, v in respaldo.items():
                    params[n].copy_(v)


# ------------------------------------------------------------------
# Métricas corregidas
# ------------------------------------------------------------------
def _probs(logits):
    return torch.softmax(torch.tensor(np.asarray(logits), dtype=torch.float32), dim=-1).numpy()


def ece_binaria(probs, labels, n_bins=10, igual_masa=True):
    conf = probs.max(axis=1)
    acierto = (probs.argmax(axis=1) == labels).astype(float)
    if igual_masa:
        orden = np.argsort(conf, kind="stable")
        grupos = np.array_split(orden, n_bins)
    else:
        bordes = np.linspace(0.0, 1.0, n_bins + 1)
        grupos = [np.where((conf > lo) & (conf <= hi))[0] for lo, hi in zip(bordes[:-1], bordes[1:])]
    return float(sum(len(g) / len(conf) * abs(acierto[g].mean() - conf[g].mean()) for g in grupos if len(g)))


def nll_binaria(probs, labels):
    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0))))


def metricas_extendidas(logits, labels):
    labels = np.asarray(labels)
    probs = _probs(logits)
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    errores = pred != labels
    return {
        "f1_macro": f1_score(labels, pred, average="macro"),
        "nll": nll_binaria(probs, labels),
        "brier": float(np.mean((probs[:, 1] - labels) ** 2)),
        "ece": ece_binaria(probs, labels, igual_masa=True),
        "ece_ancho": ece_binaria(probs, labels, igual_masa=False),
        "conf_errores": float(conf[errores].mean()) if errores.any() else np.nan,
        "frac_errores_conf90": float((errores & (conf > 0.9)).sum() / max(1, errores.sum())),
    }


def crear_compute_metrics(registro):
    """compute_metrics que además guarda los logits de validación de cada evaluación."""
    def compute(prediction):
        logits, labels = prediction
        if isinstance(logits, tuple):
            logits = logits[0]
        trainer = registro["trainer"]
        if trainer is not None and trainer.prefijo_eval.endswith("_val"):
            registro["pasos"].append(int(trainer.state.global_step))
            registro["logits"].append(np.asarray(logits, dtype=np.float32))
        return metricas_extendidas(logits, labels)
    return compute


# ------------------------------------------------------------------
# Entrenamiento de un fold
# ------------------------------------------------------------------
def _args_piloto(carpeta, cfg, seed, eval_steps, epocas, warmup):
    sig = inspect.signature(TrainingArguments.__init__).parameters
    params = dict(
        output_dir=str(carpeta), learning_rate=cfg["learning_rate"],
        per_device_train_batch_size=cfg["batch_size"], per_device_eval_batch_size=64,
        num_train_epochs=epocas, weight_decay=cfg["weight_decay"],
        optim="adamw_torch", logging_strategy="steps", logging_steps=eval_steps,
        eval_steps=eval_steps, save_strategy="no", load_best_model_at_end=False,
        report_to="none", disable_tqdm=True, dataloader_num_workers=0, seed=seed,
        data_seed=seed, bf16=USE_BF16, fp16=USE_FP16,
    )
    # transformers 5: warmup_steps acepta una fracción en [0, 1); versiones 4.x: warmup_ratio.
    if "warmup_ratio" in sig:
        params["warmup_ratio"] = float(warmup)
    else:
        params["warmup_steps"] = float(warmup)
    params["eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"] = "steps"
    descartados = sorted(k for k in params if k not in sig)
    if descartados:
        raise RuntimeError(f"TrainingArguments no acepta {descartados}: revisa la versión de transformers")
    return TrainingArguments(**params)


def subconjunto_por_grupos(tr, groups, fraccion, seed):
    """Submuestra anidada por grupos (25 % ⊂ 50 % ⊂ 75 %) con orden aleatorio fijo."""
    rng = np.random.default_rng(seed)
    grupos_tr = pd.unique(groups[tr])
    rng.shuffle(grupos_tr)
    tam = pd.Series(groups[tr]).value_counts()
    acumulado = np.cumsum([tam[g] for g in grupos_tr])
    elegidos = set(grupos_tr[: int(np.searchsorted(acumulado, fraccion * len(tr))) + 1])
    return np.sort(np.array([i for i in tr if groups[i] in elegidos]))


def huella(cfg, clave, variante, numero, tr, val):
    datos = {
        "cfg": cfg, "piloto": PILOTOS[clave], "variante": variante, "inner": numero,
        "tr": hashlib.md5(np.asarray(tr).tobytes()).hexdigest(),
        "val": hashlib.md5(np.asarray(val).tobytes()).hexdigest(),
        "modelo": MODEL_NAME, "seed": SEED, "evals_por_epoca": EVALS_POR_EPOCA, "version": "v5b-1",
        "warmup": float(PILOTOS[clave].get("warmup", WARMUP_V4_EFECTIVO)),
    }
    return json.loads(json.dumps(datos, sort_keys=True, default=str))


def entrenar_fold(clave, variante, cfg, ds, y, groups, tr, val, numero):
    d = PILOTOS[clave]
    seed_fold = SEED + SEED * 100 + numero  # misma semilla que el diagnóstico v4
    if "fraccion_train" in d:
        tr = subconjunto_por_grupos(tr, groups, d["fraccion_train"], seed_fold)
    pasos_epoca = math.ceil(len(tr) / cfg["batch_size"])
    eval_steps = max(1, pasos_epoca // EVALS_POR_EPOCA)
    epocas = int(cfg["num_train_epochs"])
    warmup = float(d.get("warmup", WARMUP_V4_EFECTIVO))
    carpeta = TMP_DIR / f"pb_{clave}_{variante}_in{numero}"

    model = crear_modelo(seed_fold, cfg["classifier_dropout"])
    detalle = ""
    if d.get("congelar_capas"):
        detalle = f"capas congeladas={congelar_capas(model, d['congelar_capas'])}"
    if d.get("dropout_interno") is not None:
        detalle = f"dropouts internos a {d['dropout_interno']}={fijar_dropout_interno(model, d['dropout_interno'])}"
    if d.get("mixout_p"):
        n_lin, n_drop = aplicar_mixout(model, d["mixout_p"])
        detalle = f"MixLinear={n_lin} | dropouts del encoder a 0={n_drop}"
    if d.get("lora_r"):
        model = aplicar_lora(model, d)
    n_total = sum(p.numel() for p in model.parameters())
    n_entr = sum(p.numel() for p in model.parameters() if p.requires_grad)

    cw = pesos_balanceados(y[tr]) if cfg["loss_mode"] == "balanced" else None
    registro = {"trainer": None, "pasos": [], "logits": []}
    ema = CallbackEMA(d["ema_beta"]) if d.get("ema_beta") else None
    ds_tr, ds_val = Subset(ds, tr.tolist()), Subset(ds, val.tolist())
    trainer = None
    try:
        trainer = entrenador_silencioso(
            TrainerPilotos(
                model=model,
                args=_args_piloto(carpeta, cfg, seed_fold, eval_steps, epocas, warmup),
                train_dataset=ds_tr,
                eval_dataset={"train": ds_tr, "val": ds_val},  # train COMPLETO en modo eval
                data_collator=data_collator,
                compute_metrics=crear_compute_metrics(registro),
                class_weights=cw,
                label_smoothing=cfg.get("label_smoothing", 0.0),
                lr_decay=cfg.get("lr_decay", 1.0),
                rdrop_alpha=d.get("rdrop_alpha", 0.0),
                callback_ema=ema,
            )
        )
        registro["trainer"] = trainer
        print(f"[CHECK] {clave} {variante.upper()} inner {numero} | n_train={len(tr)} | "
              f"entrenables={n_entr / n_total:.1%} | LR={cfg['learning_rate']} | "
              f"LS={trainer.label_smoothing} | LRD={trainer.lr_decay} | loss={cfg['loss_mode']} | "
              f"warmup={trainer.args.get_warmup_steps(trainer.args.max_steps if trainer.args.max_steps > 0 else epocas * pasos_epoca)} pasos | "
              f"R-Drop={trainer.rdrop_alpha} | EMA={d.get('ema_beta')} {detalle}")
        m_tr = trainer.evaluate(eval_dataset=ds_tr, metric_key_prefix="eval_train")
        m_va = trainer.evaluate(eval_dataset=ds_val, metric_key_prefix="eval_val")
        paso0 = {**{f"train_{k[11:]}": v for k, v in m_tr.items() if k.startswith("eval_train_")},
                 **{f"val_{k[9:]}": v for k, v in m_va.items() if k.startswith("eval_val_")}}
        trainer.state.log_history = []
        trainer.train()
        filas = {0: {"step": 0, **paso0}}
        for e in trainer.state.log_history:
            if "step" not in e:
                continue
            fila = filas.setdefault(e["step"], {"step": e["step"]})
            for k, v in e.items():
                if k.startswith("eval_train_"):
                    fila[f"train_{k[11:]}"] = float(v)
                elif k.startswith("eval_val_"):
                    fila[f"val_{k[9:]}"] = float(v)
                elif k == "loss":
                    fila["train_loss_running"] = float(v)
        tabla = pd.DataFrame(list(filas.values())).sort_values("step")
        tabla = tabla[tabla["val_f1_macro"].notna()].reset_index(drop=True)
        tabla["epoch"] = tabla["step"] / pasos_epoca
        tabla["pasos_epoca"] = pasos_epoca
        tabla["n_train"] = len(tr)
        logits = {"pasos": np.array(registro["pasos"]), "logits": np.stack(registro["logits"]),
                  "labels": y[val], "val_idx": np.asarray(val)}
    finally:
        if trainer is not None:
            del trainer
        registro["trainer"] = None
        del model
        liberar_memoria()
        shutil.rmtree(carpeta, ignore_errors=True)
    return tabla, logits


def ejecutar_pilotos_b():
    configs = json.loads((RESULTS_DIR / f"{BASE}_config_final.json").read_text(encoding="utf-8"))
    if REHACER and PILOTOS_DIR.exists():
        shutil.rmtree(PILOTOS_DIR)
    PILOTOS_DIR.mkdir(parents=True, exist_ok=True)
    total = len(VARIANTES_PILOTO) * len(PILOTOS_A_CORRER) * 3
    print(f"[PILOTOS-B] fine-tunings totales={total} (R-Drop cuesta ~2×)")

    partes, duraciones, hechos = [], [], 0
    for variante in VARIANTES_PILOTO:
        df = TRAIN_AGRUPADO[variante].copy()
        cache, y = cache_dataset(df)
        groups = df[GROUP_COL].astype(str).to_numpy()
        skf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
        splits = list(skf.split(np.zeros(len(y)), y, groups=groups))
        for clave in PILOTOS_A_CORRER:
            cfg = {k: v for k, v in configs[variante].items() if k != "refit_epochs"}
            cfg.update({k: v for k, v in PILOTOS[clave].items() if k in cfg})
            ds = cache[cfg["preprocessing"]]
            for numero, (tr, val) in enumerate(splits, start=1):
                carpeta = PILOTOS_DIR / clave
                carpeta.mkdir(parents=True, exist_ok=True)
                ruta_csv = carpeta / f"{variante}_inner{numero}.csv"
                ruta_npz = carpeta / f"{variante}_inner{numero}_logits.npz"
                ruta_json = carpeta / f"{variante}_inner{numero}_huella.json"
                h = huella(cfg, clave, variante, numero, tr, val)
                if ruta_csv.exists() and ruta_npz.exists() and ruta_json.exists():
                    if json.loads(ruta_json.read_text(encoding="utf-8")) == h:
                        partes.append(pd.read_csv(ruta_csv))
                        print(f"[PILOTOS-B] {clave} {variante.upper()} inner {numero}: reutilizado (huella OK)")
                        continue
                    print(f"[PILOTOS-B] {clave} {variante.upper()} inner {numero}: huella distinta -> reentrenando")
                t0 = time.time()
                tabla, logits = entrenar_fold(clave, variante, cfg, ds, y, groups, tr, val, numero)
                tabla.insert(0, "inner_fold", numero)
                tabla.insert(0, "variante", variante)
                tabla.insert(0, "piloto", clave)
                csv_atomico(tabla, ruta_csv)
                np.savez_compressed(ruta_npz, **logits)
                ruta_json.write_text(json.dumps(h, indent=1, ensure_ascii=False), encoding="utf-8")
                partes.append(tabla)
                duraciones.append(time.time() - t0)
                hechos += 1
                print(f"[PILOTOS-B] {clave} {variante.upper()} inner {numero}: "
                      f"{duraciones[-1] / 60:.1f} min | media {np.mean(duraciones) / 60:.1f} min/fold")
    pil = pd.concat(partes, ignore_index=True)
    pil["epoch_r"] = (pil["epoch"] * EVALS_POR_EPOCA).round() / EVALS_POR_EPOCA
    csv_atomico(pil, RESULTS_DIR / f"{BASE}_pilotosb_convergencia.csv")
    if AVISO_SONORO and os.name == "nt":
        import winsound
        winsound.Beep(880, 600)
    return pil


# ------------------------------------------------------------------
# Temperature scaling con folds cruzados (sin tocar el fold evaluado)
# ------------------------------------------------------------------
def _ajustar_T(logits, labels):
    grilla = np.exp(np.linspace(np.log(0.25), np.log(8.0), 400))
    nll = [nll_binaria(_probs(logits / t), labels) for t in grilla]
    return float(grilla[int(np.argmin(nll))])


def temperature_scaling_cruzado(clave, variante, epoca):
    datos = []
    for numero in (1, 2, 3):
        z = np.load(PILOTOS_DIR / clave / f"{variante}_inner{numero}_logits.npz")
        i = int(np.argmin(np.abs(z["pasos"] - epoca * pasos_por_epoca(clave, variante, numero))))
        datos.append((z["logits"][i], z["labels"]))
    filas = []
    for k in range(3):
        otros_z = np.concatenate([datos[j][0] for j in range(3) if j != k])
        otros_y = np.concatenate([datos[j][1] for j in range(3) if j != k])
        T = _ajustar_T(otros_z, otros_y)
        z, yk = datos[k]
        antes, despues = _probs(z), _probs(z / T)
        filas.append({"T": T, "nll_antes": nll_binaria(antes, yk), "nll_despues": nll_binaria(despues, yk),
                      "ece_antes": ece_binaria(antes, yk), "ece_despues": ece_binaria(despues, yk)})
    return pd.DataFrame(filas).mean().to_dict()


def pasos_por_epoca(clave, variante, numero):
    d = PILOTOS_RESULTADOS_B
    sel = d[(d["piloto"] == clave) & (d["variante"] == variante) & (d["inner_fold"] == numero)]
    return float(sel["pasos_epoca"].iloc[0])


# ------------------------------------------------------------------
# Resumen, decisión y figuras
# ------------------------------------------------------------------
def resumir_pilotos_b(pil):
    filas = []
    for (variante, clave), d in pil.groupby(["variante", "piloto"]):
        media = d.groupby("epoch_r", as_index=False).mean(numeric_only=True)
        i = media["val_f1_macro"].idxmax()
        ep = float(media.loc[i, "epoch_r"])
        fila = {
            "modelo": NOMBRE_EXPERIMENTO, "variante": variante, "piloto": clave,
            "descripcion": PILOTOS[clave]["nombre"], "n_train": int(d["n_train"].mean()),
            "f1_max_curva_media": media.loc[i, "val_f1_macro"],
            "f1_media_max_fold": d.groupby("inner_fold")["val_f1_macro"].max().mean(),
            "epoca_max_f1": ep,
            "train_f1_en_max": media.loc[i, "train_f1_macro"],
            "brecha_f1": media.loc[i, "train_f1_macro"] - media.loc[i, "val_f1_macro"],
            "val_nll_en_max": media.loc[i, "val_nll"],
            "train_nll_en_max": media.loc[i, "train_nll"],
            "val_nll_min": media["val_nll"].min(),
            "epoca_nll_min": float(media.loc[media["val_nll"].idxmin(), "epoch_r"]),
            "subida_val_nll": media["val_nll"].iloc[-1] - media["val_nll"].min(),
            "val_brier_en_max": media.loc[i, "val_brier"],
            "val_ece_en_max": media.loc[i, "val_ece"],
            "val_ece_ancho_en_max": media.loc[i, "val_ece_ancho"],
            "conf_errores_en_max": media.loc[i, "val_conf_errores"],
            "frac_errores_conf90_en_max": media.loc[i, "val_frac_errores_conf90"],
        }
        try:
            ts = temperature_scaling_cruzado(clave, variante, ep)
            fila.update({"T_media": ts["T"], "nll_tras_T": ts["nll_despues"], "ece_tras_T": ts["ece_despues"]})
        except Exception as e:  # noqa: BLE001
            print(f"[TS] {clave} {variante}: no calculado ({e})")
        filas.append(fila)
    res = pd.DataFrame(filas)
    if "P0" in set(res["piloto"]):
        b = res[res["piloto"] == "P0"].set_index("variante")
        for col, nombre in [("f1_max_curva_media", "dF1_curva"), ("f1_media_max_fold", "dF1_fold"),
                            ("brecha_f1", "d_brecha"), ("val_nll_en_max", "d_nll"),
                            ("val_ece_en_max", "d_ece"), ("val_brier_en_max", "d_brier")]:
            res[nombre] = res.apply(lambda r: r[col] - b.loc[r["variante"], col], axis=1)
        res["no_daña_f1"] = (res["dF1_curva"] >= -UMBRAL_F1) & (res["dF1_fold"] >= -UMBRAL_F1)
        res["mejora_brecha"] = res["d_brecha"] <= -UMBRAL_BRECHA
        res["mejora_nll"] = res["d_nll"] <= -UMBRAL_NLL
        res["candidato"] = res["no_daña_f1"] & (res["mejora_brecha"] | res["mejora_nll"])
        res.loc[res["piloto"].isin(ES_CURVA_APRENDIZAJE | {"P0"}), "candidato"] = False
    csv_atomico(res, RESULTS_DIR / f"{BASE}_pilotosb_resumen.csv")
    return res


COLORES_B = {"P0": "black", "P1": "C0", "P4": "C1", "P5": "C2", "P6": "C3", "P7": "C4", "P8": "C5", "P9": "C6", "P10": "C7", "P11": "C8", "P12": "C9",
             "LC25": "#c6dbef", "LC50": "#6baed6", "LC75": "#2171b5"}


def figura_pilotos_b(pil, claves, sufijo, titulo):
    variantes = [v for v in ["mx", "es", "cu"] if v in set(pil["variante"])]
    filas = [("nll", "NLL sin pesos (modo eval)"), ("f1_macro", "F1-Macro"), ("ece", "ECE (igual masa)")]
    fig, axes = plt.subplots(len(filas), len(variantes), figsize=(5.4 * len(variantes), 10.5),
                             sharex="col", squeeze=False)
    for j, variante in enumerate(variantes):
        for clave in [c for c in claves if c in set(pil["piloto"])]:
            d = pil[(pil["variante"] == variante) & (pil["piloto"] == clave)]
            media = d.groupby("epoch_r", as_index=False).mean(numeric_only=True)
            for fi, (met, _) in enumerate(filas):
                ax = axes[fi, j]
                ax.plot(media["epoch_r"], media[f"val_{met}"], color=COLORES_B[clave], lw=2.0,
                        label=f"{clave}: {PILOTOS[clave]['nombre']}")
                if met != "ece":
                    ax.plot(media["epoch_r"], media[f"train_{met}"], color=COLORES_B[clave], lw=0.9,
                            ls="--", alpha=0.6)
        axes[0, j].set_title(variante.upper(), fontsize=11)
        axes[-1, j].set_xlabel("Época")
        for ax in axes[:, j]:
            ax.grid(alpha=0.25)
    for fi, (_, etiqueta) in enumerate(filas):
        axes[fi, 0].set_ylabel(etiqueta)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=3, fontsize=8, frameon=False)
    fig.suptitle(f"{NOMBRE_EXPERIMENTO} — {titulo}\n(media de 3 Inner Folds; continua = validación, "
                 "discontinua = train completo)", fontsize=10.5)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    for ext in ("png", "pdf"):
        fig.savefig(RESULTS_DIR / f"{BASE}_pilotosb_{sufijo}.{ext}", dpi=200, bbox_inches="tight")
    display(fig)
    plt.close(fig)


if not (RESULTS_DIR / f"{BASE}_config_final.json").exists():
    raise FileNotFoundError(f"Falta {BASE}_config_final.json en {RESULTS_DIR}.")
PILOTOS_RESULTADOS_B = ejecutar_pilotos_b()
PILOTOS_RESUMEN_B = resumir_pilotos_b(PILOTOS_RESULTADOS_B)
cols = ["variante", "piloto", "descripcion", "f1_max_curva_media", "f1_media_max_fold", "brecha_f1",
        "val_nll_en_max", "val_ece_en_max", "dF1_curva", "dF1_fold", "d_brecha", "d_nll", "candidato"]
display(PILOTOS_RESUMEN_B[[c for c in cols if c in PILOTOS_RESUMEN_B]].round(4))
figura_pilotos_b(PILOTOS_RESULTADOS_B, ["P0", "P1", "P4", "P5", "P6", "P7", "P8", "P9", "P10", "P11", "P12"], "tecnicas",
                 "pilotos de regularización (ronda 2)")
figura_pilotos_b(PILOTOS_RESULTADOS_B, ["LC25", "LC50", "LC75", "P0"], "curva_aprendizaje",
                 "curva de aprendizaje (25 %, 50 %, 75 %, 100 % del Inner Train)")
print(f"✅ [{NOMBRE_EXPERIMENTO}] Pilotos ronda 2 terminados — {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"Resultados en: {PILOTOS_DIR.resolve()}")


## 9. Qué mandar de vuelta

Copia de vuelta a la Mac la carpeta `data/resultados_<modelo>_grouped_5x3_tpe20_v4/` completa (incluida `pilotos_v5b/`, que trae los logits para el temperature scaling).
